# Lecture 16: Spacecraft trajectories and the primer vector

**The challenge.** Spaceflight is where the indirect method grew up, and it
is still where it is used most. The problems combine everything so far. The
thrust *direction* is a unit vector, the throttle is bang-bang, gravity
makes the dynamics stiffly nonlinear, and costs are often Mayer form (the
final mass, the final orbit radius). Terminal conditions describe a whole
manifold (any point on a target orbit) or a moving target (a planet at a
given date), and the units are astronomical. This lecture solves two
classics, and shows what the necessary conditions say about rockets in
general, through Lawden's **primer vector**.

**After this lecture you can**

- pose orbit-transfer problems with unit-vector controls, Mayer costs,
  terminal manifolds and normalized units;
- explain the primer vector and read thrust direction and throttle from it;
- solve maximum-radius and fuel-optimal transfers from a costate box, and
  check them against the literature.

## 1. An architect, a primer and an ion engine

In 1925 Walter Hohmann, the city architect of Essen and an amateur of
spaceflight, published *Die Erreichbarkeit der Himmelskörper* ("The
attainability of celestial bodies"). In it he showed that the cheapest way
between two circular orbits, with impulsive burns, is two burns half an
orbit apart: the **Hohmann transfer**, still the first thing taught in
astrodynamics.

The general theory of optimal rocket trajectories came from the calculus of
variations. Derek Lawden, a British mathematician, showed in the 1950s that
in an optimal trajectory the thrust always points along a vector built from
the velocity costates, which he called the **primer vector**. He collected
the theory in *Optimal Trajectories for Space Navigation* (1963). He also
found an intermediate-thrust arc in space, "Lawden's spiral". In 1965 H. M.
Robbins showed, with the second-order test of Lecture 12, that it is not a
minimum. Singular arcs had to be checked, not just found.

Electric propulsion made low thrust practical: engines that push with a fraction of
a newton for months, with ten times the efficiency of
chemical rockets. NASA's *Deep Space 1* flew the first ion engine on an
interplanetary mission in 1998, and *Dawn* used ion propulsion to orbit both
Vesta and Ceres. Their trajectories coast and burn for years, which is
exactly the fuel-optimal bang-bang structure of Lecture 10.

## 2. The primer vector

Write the dynamics as $\dot r = v$, $\dot v = g(r) + (T\delta/m)\,\hat a$,
$\dot m = -T\delta/c$, with $\hat a$ a unit vector and $\delta \in [0, 1]$.
The Hamiltonian is

$$
H = L + \lambda_r^\top v + \lambda_v^\top g(r) + \frac{T\delta}{m}\lambda_v^\top\hat a - \lambda_m\frac{T\delta}{c} .
$$

Minimizing over the unit vector (Lecture 4) gives
$\hat a = -\lambda_v / \lVert\lambda_v\rVert$: thrust along the **primer vector**
$p = -\lambda_v$. What is left is linear in $\delta$. For the fuel-optimal
cost $L = T\delta/c$, its coefficient is the switching function

$$
S = \frac{T}{c}\Big(1 - \frac{c\,\lVert p\rVert}{m} - \lambda_m\Big),
$$

so the engine fires when the primer is long enough, $\lVert p\rVert >
m(1 - \lambda_m)/c$, and coasts otherwise. Lawden's rules in one picture:
point along the primer, fire when it is long. The package derives exactly
this for its low-thrust example (Lecture 4). Here it is for a full
three-dimensional problem.

In [ ]:
# Lawden's switching rule (a sketch): the engine fires exactly while the primer's length
# |p| is above the threshold m (1 - lambda_m) / c, that is, while S < 0.
import matplotlib.pyplot as plt
import numpy as np

BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#8a8a85"
fig, ax = plt.subplots(figsize=(8, 3.2))
t = np.linspace(0, 10, 600)
p_norm = 1.0 + 0.45 * np.cos(1.3 * t) * np.exp(-0.03 * t) + 0.05 * t
threshold = 1.05 + 0.02 * t                        # slowly varying, as m and lambda_m change
burn = p_norm > threshold
ax.fill_between(t, 0, 2.2, where=burn, color=ORANGE, alpha=0.15, lw=0, label="engine on")
ax.plot(t, p_norm, color=BLUE, lw=2, label=r"primer length $\|p\|$")
ax.plot(t, threshold, color="k", lw=1, ls="--", label=r"threshold $m(1 - \lambda_m)/c$")
ax.set(xlabel="t", ylim=(0.4, 2.05), yticks=[])
ax.legend(frameon=False, fontsize=8, loc="upper left", ncol=3)
ax.set_title("burn while the primer is long, coast while it is short", fontsize=10, loc="left")
ax.grid(alpha=0.25)
fig.tight_layout()

*Lawden's switching rule, sketched. The engine fires exactly while the primer's length is above the threshold, which is where $S < 0$. Section 4 shows the real primer of an Earth–Mars transfer.*

## 3. The largest orbit in a given time

Bryson and Ho's orbit-raising problem: a spacecraft with constant thrust,
losing mass at a constant rate, starts on a circular orbit and has a fixed
time to reach the largest possible circular orbit. In polar coordinates,
normalized so that $\mu = 1$ and the initial radius is 1:

$$
\dot r = v_r,\quad \dot\theta = \frac{v_\theta}{r},\quad
\dot v_r = \frac{v_\theta^2}{r} - \frac{\mu}{r^2} + a\,u_1,\quad
\dot v_\theta = -\frac{v_r v_\theta}{r} + a\,u_2,\quad
a = \frac{T}{m_0 - \dot m\,t},
$$

with $(u_1, u_2)$ a unit vector. The cost is Mayer, $\phi = -r(t_f)$ (maximize
the radius), and the terminal conditions say the final orbit is circular:
$v_r = 0$, $v_\theta = \sqrt{\mu/r}$. The angle $\theta$ is free at the end,
so its costate vanishes there, and being cyclic it vanishes throughout.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import sympy as sp

from auto_optimal_control.homotopy import Homotopy
from auto_optimal_control.pontryagin import derive
from auto_optimal_control.problem import Box, OptimalControlProblem, UnitVector

BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#8a8a85"

t = sp.Symbol("t", real=True)
r = sp.Symbol("r", positive=True)
th, vr, vt, u1, u2 = sp.symbols("theta v_r v_theta u_1 u_2", real=True)
mu, T, m0, mdot, tf = sp.symbols("mu T m_0 mdot t_f", positive=True)
accel = T / (m0 - mdot * t)
orbit_raising = OptimalControlProblem(
    name="orbit_raising", time=t, states=[r, th, vr, vt], controls=[u1, u2],
    params=[mu, T, m0, mdot, tf],
    dynamics=[vr, vt / r, vt**2 / r - mu / r**2 + accel * u1, -vr * vt / r + accel * u2],
    terminal_cost=-r, terminal_constraints=[vr, vt - sp.sqrt(mu / r)], final_time=tf,
    initial_state=[1, 0, 0, sp.sqrt(mu)], control_sets=[UnitVector([u1, u2])],
)
print("thrust direction law:", derive(orbit_raising).control_law)
p_or = [1.0, 0.1405, 1.0, 0.0749, 3.32]               # Bryson & Ho's values
ro = Homotopy(orbit_raising).solve(p_or, ([-2] * 4, [2] * 4))
tr = ro.shooting.trajectory(ro.solution.v, n_points=400)
print(f"r(t_f) = {tr['z'][-1, 0]:.10f}  (reference 1.5252776, Bryson & Ho give 1.525)")

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4))
x_, y_ = tr["z"][:, 0] * np.cos(tr["z"][:, 1]), tr["z"][:, 0] * np.sin(tr["z"][:, 1])
left.plot(x_, y_, color=BLUE, lw=2)
for i in range(0, len(tr["t"]), 25):
    rad, ang = tr["z"][i, 0], tr["z"][i, 1]
    er, et = np.array([np.cos(ang), np.sin(ang)]), np.array([-np.sin(ang), np.cos(ang)])
    d = tr["u"][i, 0] * er + tr["u"][i, 1] * et
    left.arrow(x_[i], y_[i], 0.15 * d[0], 0.15 * d[1], head_width=0.03, color=ORANGE)
ang = np.linspace(0, 2 * np.pi, 200)
for rad in (1.0, tr["z"][-1, 0]):
    left.plot(rad * np.cos(ang), rad * np.sin(ang), "k--", lw=0.5)
left.set_aspect("equal")
left.set_title("path and thrust direction", fontsize=10, loc="left")
angle = np.degrees(np.unwrap(np.arctan2(tr["u"][:, 0], tr["u"][:, 1])))
right.plot(tr["t"], angle, color=ORANGE, lw=2)
right.axhline(90, color=GRAY, lw=0.8, ls=":")
right.axhline(270, color=GRAY, lw=0.8, ls=":")
right.set(xlabel="t", ylabel="thrust angle from the local horizontal (deg)")
for a_ in (left, right):
    a_.grid(alpha=0.3)
fig.tight_layout()

The thrust direction turns through most of a circle. For the first half it
points outward and forward, turning slowly towards straight out ($90°$),
raising the orbit. Around mid-flight it swings quickly through backward
($180°$): a larger circular orbit is *slower*, so the excess tangential
speed must go. At the end it points inward and forward
(past $270°$), killing the radial velocity so the spacecraft arrives on a
circle. That is the textbook picture in Bryson and Ho, here to ten digits.

The law `derive` printed contains `Abs(m_0 - mdot*t)/(-m_0 + mdot*t)`, which
is just $-1$ while the spacecraft has mass left. SymPy cannot know that
$m_0 > \dot m t$, because only symbols, not expressions, can be declared
positive. The generated code evaluates it correctly anyway; making the mass
a state (as in the next problem) avoids it.

## 4. Earth to Mars on a fuel budget

The second classic is a full rendezvous: a 1000 kg spacecraft with a 0.5 N
engine (specific impulse 2000 s) leaves Earth and must arrive at Mars, with
Mars's position and velocity, after 348.795 days, using as little propellant
as possible. This is the benchmark of Taheri and colleagues (2024), in
three-dimensional Cartesian coordinates, with distances in astronomical
units and time in years (so $\mu_\odot \approx 4\pi^2$). The throttle is
bang-bang and the direction follows the primer. The planets' states are
parameters, so one compiled problem serves any launch date.

In [ ]:
AU, TU = 1.496e8, 3.1536e7                           # km, s
MARS = dict(
    mu=132712440018.0 * TU**2 / AU**3, m0=1000.0,
    T=0.5 / 1000.0 / 1000.0 * TU**2 / AU,            # 0.5 N on 1000 kg, in AU / TU^2
    c=2000.0 * 9.80665 / 1000.0 * TU / AU,           # Isp 2000 s
    rE=np.array([-140699693.0, -51614428.0, 980.0]) / AU,
    vE=np.array([9.774596, -28.07828, 4.337725e-4]) * TU / AU,
    rM=np.array([-172682023.0, 176959469.0, 7948912.0]) / AU,
    vM=np.array([-16.427384, -14.860506, 9.21486e-2]) * TU / AU,
    tf=348.795 * 86400.0 / TU,
)
rr = sp.symbols("r_x r_y r_z", real=True)
vv = sp.symbols("v_x v_y v_z", real=True)
m = sp.Symbol("m", positive=True)
aa = sp.symbols("a_x a_y a_z", real=True)
delta = sp.Symbol("delta", real=True)
mus, Ts, cs, tfs = sp.symbols("mu T_max c t_f", positive=True)
x0s, xfs = sp.symbols("x0_1:8", real=True), sp.symbols("xf_1:7", real=True)
rn3 = (rr[0] ** 2 + rr[1] ** 2 + rr[2] ** 2) ** sp.Rational(3, 2)
earth_mars = OptimalControlProblem(
    name="earth_mars_fuel", time=t, states=[*rr, *vv, m], controls=[*aa, delta],
    params=[mus, Ts, cs, tfs, *x0s, *xfs],
    dynamics=list(vv) + [-mus * rr[i] / rn3 + Ts * delta * aa[i] / m for i in range(3)]
    + [-Ts * delta / cs],
    running_cost=Ts / cs * delta, initial_state=list(x0s),
    final_state=dict(zip([*rr, *vv], xfs)), final_time=tfs,
    control_sets=[UnitVector(aa), Box(delta, 0, 1)],
)
x0 = np.concatenate([MARS["rE"], MARS["vE"], [1.0]])
p_em = [MARS["mu"], MARS["T"], MARS["c"], MARS["tf"], *x0, *MARS["rM"], *MARS["vM"]]
em = Homotopy(earth_mars).solve(p_em, ([0] * 7, [1] * 7))
k_em = em.shooting.kernels
fin = k_em.propagate(em.solution.t0, em.solution.z0, em.solution.tf, em.params)
print(em.message)
print(f"arrival mass {fin['z'][6] * MARS['m0']:.4f} kg  (Taheri et al.: 603.935 kg, smoothed)")
print("switches (days):", np.round(fin["switch_times"] * TU / 86400.0, 2))
print(f"smoothing lowered to eps = {em.steps[-1].eps:.1e} in {len(em.steps)} steps")

In [ ]:
tr = em.shooting.trajectory(em.solution.v, n_points=2000)
z = tr["z"]
primer = np.linalg.norm(z[:, 10:13], axis=1)
threshold = z[:, 6] * (1 - z[:, 13]) / MARS["c"]
fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4.2))
burn = tr["u"][:, 3] > 0.5
left.plot(z[~burn, 0], z[~burn, 1], ".", color=GRAY, ms=1)
left.plot(z[burn, 0], z[burn, 1], ".", color=ORANGE, ms=1.5)
left.plot(0, 0, "o", color="#f2b705", ms=10)
left.plot(*MARS["rE"][:2], "o", color=BLUE, ms=6)
left.plot(*MARS["rM"][:2], "o", color="#c1440e", ms=6)
left.set_aspect("equal")
left.set_title("burn (orange) and coast (gray), AU", fontsize=10, loc="left")
days = tr["t"] * TU / 86400.0
right.plot(days, primer, color=BLUE, lw=2, label=r"primer length $\|\lambda_v\|$")
right.plot(days, threshold, color="k", lw=1, ls="--", label=r"threshold $m(1 - \lambda_m)/c$")
right.fill_between(days, 0, primer.max() * 1.05, where=burn, color=ORANGE, alpha=0.15)
right.set(xlabel="days", ylim=(0, primer.max() * 1.05))
right.legend(frameon=False, fontsize=8)
for a_ in (left, right):
    a_.grid(alpha=0.3)
fig.tight_layout()

The engine burns exactly when the primer is longer than the threshold:
Lawden's rule, read off the computed extremal. The exact bang-bang solution
keeps slightly more mass than the reference, which was computed with a
smoothed throttle, as it should (a smoothed throttle wastes propellant on
partial burns). The automatic homotopy got there from the costate box of the
paper, $[0, 1]^7$; `em.steps` shows how far it had to lower the smoothing.

**Working with real units.** Normalizing distances by the astronomical unit
and time by the year makes every state and costate of order one, which the
integrator's tolerances and the random costate box both assume. Without it,
positions in kilometres and times in seconds differ by eleven orders of
magnitude, and neither tolerances nor costate boxes mean anything.

## Summary

- Thrust direction follows the primer vector $-\lambda_v$; the throttle is
  bang-bang on $S$, firing when the primer is longer than
  $m(1 - \lambda_m)/c$. Lawden's spiral is a singular arc that fails the
  second-order test.
- Orbit problems use `UnitVector` controls, Mayer costs
  (`terminal_cost=-r`), terminal manifolds (`terminal_constraints`), fixed
  final times as parameters, and normalized units.
- The package reproduces Bryson and Ho's orbit raising and the Earth–Mars
  benchmark from costate boxes alone.

## Exercises

1. In the orbit-raising problem, double the flight time. How many
   revolutions does the transfer make, and how does the thrust angle look?
2. For Earth–Mars, change the flight time by $\pm 20$ days (`t_f` is a
   parameter) and plot the arrival mass against it.
3. Make the Earth–Mars cost minimum *energy*,
   $\int (T\delta)^2/(2c)$ (no bang-bang), and compare the throttle with the
   fuel-optimal one. Which problem was easier to solve, and why?

## Further reading

- D. F. Lawden, *Optimal Trajectories for Space Navigation* (Butterworths,
  1963).
- W. Hohmann, *Die Erreichbarkeit der Himmelskörper* (Oldenbourg, 1925;
  NASA translation TT F-44, 1960).
- A. E. Bryson and Y.-C. Ho, *Applied Optimal Control*, section 2.5:
  maximum-radius orbit transfer.
- E. Taheri and coauthors, benchmark low-thrust trajectory problems,
  arXiv:2409.01490 (2024).
- J. T. Betts, "Survey of numerical methods for trajectory optimization",
  *Journal of Guidance, Control, and Dynamics* 21(2), 1998.